# MA1DCNN Architecture — Phase 3 (Architecture Only, Synthetic Data)

Builds and tests the real MA1DCNN model class, per Akindoju (2025),
Section 3.4.1 and Table 4.4 — a 1D-CNN with two specific attention
modules (**CAM**, **EAM**), not the generic "multi-head attention"
placeholder the repo's stub docstring and `configs/baseline.yaml`
originally assumed. See `ma1dcnn_architecture_plan.docx` for the full
element-by-element explanation and the math behind every width, kernel,
and channel number used below — this notebook is that plan turned into
running, tested code.

## Build scope for this notebook (agreed decision)

This notebook builds and tests **only the architecture** — the
convolutional blocks, CAM, EAM, and the fully-connected output head —
against small synthetic tensors. The training loop itself (loss
function, optimizer, scheduler, early stopping, metric computation) is
deliberately **not** built here; it is `training/train.py`'s job, once
this architecture is verified. Keeping the two separate means that if
something does not behave as expected later, it is clear whether the
problem lives in the network's structure or in how it is trained — the
same discipline `loaders.py` and `preprocessing.py` were each built and
proven correct on their own before being combined.

## What every section below does

1. **`Conv1dSamePadding`** — a 1D convolution that keeps the sequence
   length unchanged (Section 3.1 of the architecture plan).
2. **`ChannelAttentionModule` (CAM)** — learns which feature *channels*
   matter most, and reweights them.
3. **`ElementwiseAttentionModule` (EAM)** — learns which *timesteps*
   matter most, and reweights them.
4. **`MA1DCNN`** — the full model: 3 conv blocks → CAM → EAM → global
   average pooling → fully-connected head → 5 raw logits (one per fault
   component).
5. **Parameter-count and receptive-field checks** that confirm, in code,
   the exact numbers `ma1dcnn_architecture_plan.docx` Section 3 derived
   by hand.

Every class and function below has its own test cell directly after it,
built on small random tensors — no real N-CMAPSS data, Drive, or
windowing pipeline is needed to run or understand any single piece of
this notebook.

## 0. Imports

Only `torch` and `torch.nn` are needed to build the architecture itself
— no numpy, no file I/O, since this notebook never touches real data.

In [1]:
import torch          # the tensor/autograd library everything below is built on
import torch.nn as nn  # nn.Module, nn.Conv1d, nn.Linear, etc. -- the building blocks of every class below

## 1. Constants

Every architectural judgment call — the output-column order, the
illustrative input-channel count, the CAM bottleneck ratio, and the two
concrete hyperparameter sets from Akindoju's Table 4.4 — is named here,
once, rather than buried inside a class body.

In [2]:
# Fixed output-column order for the label matrix -- MUST match loaders.py's
# FAULT_COLUMNS exactly, since this list's length (5) is what fixes the
# model's output layer width below. Repeated here (not imported) so this
# notebook can run completely standalone, same convention preprocessing.ipynb
# already uses for NCMAPSS_FILES.
FAULT_COLUMNS = ["fan_fail", "lpc_fail", "hpc_fail", "hpt_fail", "lpt_fail"]
N_FAULT_COLUMNS = len(FAULT_COLUMNS)  # 5 -- the output layer's width; never a free hyperparameter

# Illustrative input-channel count for this architecture-only pass: the
# 14 X_s (physical sensor) + 14 X_v (virtual/derived sensor) channels
# preprocessing.py normalizes. W (4 flight-condition columns: altitude,
# Mach number, throttle-resolver angle, T2) is ALSO normalized in
# preprocessing.py, but is deliberately left OUT of this count here --
# W describes flight condition, not an engine-health signal that varies
# along the windowed time axis the same way a vibration or temperature
# sensor does. Whether W should later be concatenated as extra channels,
# or handled separately, is a decision for the not-yet-built windowing
# notebook (see ma1dcnn_architecture_plan.docx, Section 5, Step 1) --
# flagged here as an assumption, not silently decided here.
DEFAULT_N_INPUT_CHANNELS = 28

# Channel-attention (CAM) bottleneck reduction ratio: CAM's squeeze step
# shrinks channel_3 channels down to channel_3 // CAM_REDUCTION_RATIO
# before expanding back. This is a standard squeeze-and-excite-style
# judgment call (not one of Akindoju's tuned numbers) -- picked so the
# bottleneck (32 units when channel_3=128) is small enough to force real
# compression, without collapsing to 1-2 units.
CAM_REDUCTION_RATIO = 4

# Table 4.4 (Akindoju, 2025) -- "Best Trial (Before Tuning)" column.
# This is Optuna's untuned starting point, not a guess.
BEFORE_TUNING_CONFIG = {
    "n_input_channels": DEFAULT_N_INPUT_CHANNELS,  # placeholder pending the windowing notebook -- see note above
    "channel_1": 32,      # Table 4.4: Channel 1
    "channel_2": 64,      # Table 4.4: Channel 2 (before tuning)
    "channel_3": 128,     # Table 4.4: Channel 3
    "kernel_size": 7,     # Table 4.4: Kernel (before tuning)
    "fc_units": 64,       # Table 4.4: fc_units (before tuning)
    "dropout": 0.5,       # Table 4.4: Dropout (before tuning)
    "n_outputs": N_FAULT_COLUMNS,  # fixed at 5 -- not a tunable hyperparameter
}

# Table 4.4 -- "Best Trial (After Tuning)" column: what Optuna's search
# actually converged on.
AFTER_TUNING_CONFIG = {
    "n_input_channels": DEFAULT_N_INPUT_CHANNELS,
    "channel_1": 32,          # unchanged by tuning
    "channel_2": 128,         # Table 4.4: Channel 2 (after tuning) -- doubled from 64
    "channel_3": 128,         # unchanged by tuning
    "kernel_size": 5,         # Table 4.4: Kernel (after tuning) -- shrunk from 7
    "fc_units": 16,           # Table 4.4: fc_units (after tuning) -- shrunk from 64
    "dropout": 0.39919,       # Table 4.4: Dropout (after tuning)
    "n_outputs": N_FAULT_COLUMNS,
}

# New judgment call (not from Table 4.4): same as BEFORE_TUNING_CONFIG, but adds dropout
# inside the conv stack itself -- currently MA1DCNN applies dropout exactly once, right
# before the FC layer, so all three conv blocks have zero regularization pressure. Added
# after the real Oct 2026 runs (train.ipynb Section 10) showed neither coupled nor
# decoupled weight_decay alone closed the train/val generalization gap -- this targets
# the feature-extraction stage directly instead of penalizing every weight uniformly.
# conv_dropout=0.0 is the default on Conv1dSamePadding/MA1DCNN/build_ma1dcnn, so
# BEFORE_TUNING_CONFIG and AFTER_TUNING_CONFIG above build byte-identical models to
# before this change -- this new config is strictly additive.
BEFORE_TUNING_CONFIG_CONV_DROPOUT = dict(BEFORE_TUNING_CONFIG, conv_dropout=0.2)

# Next single-variable step (see 'What's next' below): BatchNorm1d after each conv
# block, LAYERED ON TOP of conv-stack dropout, not replacing it -- normalizing each
# conv block's output before the nonlinearity is a different lever than zeroing some
# of its activations, and the two are not mutually exclusive. use_batchnorm=False is
# the default on Conv1dSamePadding/MA1DCNN/build_ma1dcnn, so every config above
# (including BEFORE_TUNING_CONFIG_CONV_DROPOUT itself) builds a byte-identical model
# to before this change -- this new config is strictly additive, same discipline as
# conv_dropout before it.
BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN = dict(BEFORE_TUNING_CONFIG_CONV_DROPOUT, use_batchnorm=True)

## 2. `Conv1dSamePadding` — a conv layer that never changes the sequence length

Per Section 3.1 of `ma1dcnn_architecture_plan.docx`: for stride 1 and an
odd kernel size `K`, padding `P = (K - 1) / 2` on each side makes
`L_out = L_in` exactly. This class computes that `P` once, at
construction time, so every conv block in `MA1DCNN` below can be built
without re-deriving the formula each time.

In [3]:
class Conv1dSamePadding(nn.Module):
    """1D convolution that keeps the sequence length unchanged (stride=1, odd kernel)."""

    def __init__(self, in_channels, out_channels, kernel_size, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()  # required first call for any nn.Module subclass -- registers this module's parameters correctly

        if kernel_size % 2 == 0:
            # Even kernels have no integer P that makes L_out exactly
            # L_in for stride 1 -- fail loudly here rather than silently
            # producing a shifted or mismatched sequence length later.
            raise ValueError(f"Conv1dSamePadding needs an odd kernel_size, got {kernel_size}")

        padding = (kernel_size - 1) // 2  # the same-padding formula from Section 3.1: P = (K - 1) / 2

        self.conv = nn.Conv1d(
            in_channels=in_channels,    # number of input feature channels (e.g. 28 sensor channels, or 32/64/128 mid-network)
            out_channels=out_channels,  # number of learned filters -- becomes the channel count of the output
            kernel_size=kernel_size,    # how many consecutive timesteps each filter looks at
            stride=1,                   # stride=1 is required for the same-padding formula above to hold exactly
            padding=padding,            # computed above -- pads both ends of the sequence so length is preserved
        )
        # NEW: BatchNorm1d, placed BEFORE the activation (the standard order -- normalizing
        # the conv's raw output is what stabilizes a deep stack; normalizing AFTER ReLU would
        # normalize an already-clipped, non-negative-only distribution instead). When disabled
        # (the default), nn.Identity() contributes exactly zero parameters and zero buffers --
        # not just "behaves like a no-op" but literally no new state exists on the module at all.
        self.bn = nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
        self.activation = nn.ReLU()  # standard nonlinearity after a conv layer -- without it, stacked convs collapse to one linear map
        # dropout inside the conv stack. p=0.0 (the default) makes this an identity
        # function in BOTH train and eval mode, so every existing config is unaffected.
        self.conv_dropout = nn.Dropout(p=conv_dropout)

    def forward(self, x):
        # x has shape (batch, in_channels, L) -- PyTorch's Conv1d expects
        # channels BEFORE the sequence dimension, unlike the (rows,
        # channels) layout preprocessing.py's saved .npz arrays use.
        x = self.conv(x)            # shape (batch, out_channels, L) -- L unchanged by construction, per the padding above
        x = self.bn(x)              # NEW -- identity when use_batchnorm=False; else per-channel normalize using batch stats (train) or running stats (eval)
        x = self.activation(x)      # same shape, values clipped to >= 0
        x = self.conv_dropout(x)    # no-op when conv_dropout=0.0; zeroes activations at that rate, training only, otherwise
        return x

In [4]:
def _test_conv1d_same_padding_preserves_length():
    # Every kernel size actually used by Table 4.4 (7 before tuning, 5
    # after tuning), plus a couple of other odd sizes, should all leave
    # the sequence length exactly unchanged -- this IS the "same
    # padding" guarantee Section 3.1 of the architecture plan proves on
    # paper; this test proves it in running code.
    batch, in_channels, length = 4, 28, 50
    x = torch.randn(batch, in_channels, length)  # random values -- only the SHAPE is being checked here, not the values

    for kernel_size in [3, 5, 7, 9]:
        layer = Conv1dSamePadding(in_channels, out_channels=32, kernel_size=kernel_size)
        out = layer(x)
        assert out.shape == (batch, 32, length), (kernel_size, out.shape)  # length must match L_in exactly, for every K

    # An even kernel has no exact "same" padding -- must fail loudly,
    # not silently produce a shifted-length output.
    raised = False
    try:
        Conv1dSamePadding(in_channels, out_channels=32, kernel_size=6)
    except ValueError:
        raised = True  # expected path
    assert raised, "even kernel_size should have raised ValueError"

    print("Conv1dSamePadding: PASSED (length preserved for K=3,5,7,9; even K correctly rejected)")


_test_conv1d_same_padding_preserves_length()


def _test_conv1d_same_padding_conv_dropout_default_is_noop_and_eval_mode_disables_it():
    # Regression check for the conv_dropout parameter added above: the default (0.0) must
    # leave every existing config's behavior byte-identical, and a nonzero rate must still
    # become the identity function in eval() mode (same contract as the existing FC dropout).
    torch.manual_seed(0)
    x = torch.randn(4, 28, 50)

    default_layer = Conv1dSamePadding(28, 32, kernel_size=7)  # conv_dropout not passed -- must default to 0.0
    assert default_layer.conv_dropout.p == 0.0, default_layer.conv_dropout.p
    default_layer.eval()
    out_a = default_layer(x)
    out_b = default_layer(x)
    assert torch.equal(out_a, out_b), "conv_dropout=0.0 should be fully deterministic even without eval()"

    dropout_layer = Conv1dSamePadding(28, 32, kernel_size=7, conv_dropout=0.5)
    dropout_layer.eval()  # eval mode -- dropout must be OFF regardless of the configured rate
    out_eval_a = dropout_layer(x)
    out_eval_b = dropout_layer(x)
    assert torch.equal(out_eval_a, out_eval_b), "conv_dropout must be deterministic (OFF) in eval() mode"

    dropout_layer.train()  # train mode -- dropout must actually zero some activations
    out_train = dropout_layer(x)
    assert (out_train == 0).float().mean().item() > 0.05, "train-mode conv_dropout=0.5 should zero a meaningful fraction of activations"

    print("Conv1dSamePadding conv_dropout: PASSED (default is a no-op, eval() disables it, train() actually zeroes activations)")


_test_conv1d_same_padding_conv_dropout_default_is_noop_and_eval_mode_disables_it()

def _test_conv1d_same_padding_batchnorm_default_is_identity_and_wiring_order():
    # Regression + wiring check for the use_batchnorm parameter added above. Two things,
    # deliberately checked separately from the running-statistics test below:
    #   (a) the default (False) must leave every existing config byte-identical -- not just
    #       "behaves like a no-op" the way conv_dropout=0.0 does, but LITERALLY zero new
    #       parameters and zero new buffers, since nn.Identity() has none.
    #   (b) the layer must be sized on OUT_channels (the thing actually flowing through
    #       self.bn), not in_channels -- an easy mix-up since Conv1dSamePadding's own
    #       constructor takes both.
    torch.manual_seed(0)
    x = torch.randn(4, 28, 50)

    default_layer = Conv1dSamePadding(28, 32, kernel_size=7)  # use_batchnorm not passed -- must default to False
    assert isinstance(default_layer.bn, nn.Identity), type(default_layer.bn)
    assert list(default_layer.bn.parameters()) == [], "nn.Identity must contribute zero parameters"
    assert list(default_layer.bn.buffers()) == [], "nn.Identity must contribute zero buffers -- no running stats to accidentally save/load"
    default_layer.eval()
    out_a = default_layer(x)
    out_b = default_layer(x)
    assert torch.equal(out_a, out_b), "use_batchnorm=False should be fully deterministic even without eval()"

    bn_layer = Conv1dSamePadding(28, 32, kernel_size=7, use_batchnorm=True)
    assert isinstance(bn_layer.bn, nn.BatchNorm1d), type(bn_layer.bn)
    assert bn_layer.bn.num_features == 32, ("BatchNorm1d must track OUT_channels (32), not in_channels (28)", bn_layer.bn.num_features)

    print("Conv1dSamePadding batchnorm wiring: PASSED (default is nn.Identity with zero added state; enabled layer is BatchNorm1d(out_channels))")


_test_conv1d_same_padding_batchnorm_default_is_identity_and_wiring_order()


def _test_batchnorm_running_statistics_train_vs_eval():
    # This is the test the project's "What's next" note explicitly asked for, and it is
    # deliberately a SEPARATE test from the one above: the wiring test only has to show the
    # default is a no-op. BatchNorm1d's actual contract is a different, stateful thing
    # dropout never had -- forward() is NOT a pure function of its input in train() mode; it
    # mutates running_mean/running_var as a side effect (an exponential moving average,
    # default momentum=0.1), normalizes using the CURRENT BATCH's own statistics while doing
    # so, then in eval() mode switches to using those frozen running stats instead and stops
    # updating them. None of that is exercised by a shape check -- it would ship silently
    # wrong (e.g. normalizing with stale running stats during training, or quietly drifting
    # the running stats during validation) without a test that watches the buffers directly.
    torch.manual_seed(0)
    channels = 32
    bn = nn.BatchNorm1d(channels)

    # 1. Freshly constructed: PyTorch's documented initialization (mean=0, var=1, zero
    #    batches tracked) -- the baseline every assertion below moves away from.
    assert torch.equal(bn.running_mean, torch.zeros(channels)), "running_mean must start at 0"
    assert torch.equal(bn.running_var, torch.ones(channels)), "running_var must start at 1"
    assert bn.num_batches_tracked.item() == 0

    # 2. train() mode: a forward pass must (a) advance num_batches_tracked, (b) move the
    #    running stats away from their init, and (c) normalize the OUTPUT using this batch's
    #    OWN mean/std (which are still 0/1 at this point) -- not the running ones.
    bn.train()
    x1 = torch.randn(16, channels, 20) * 5.0 + 3.0  # mean ~3, std ~5 -- deliberately far from the 0/1 init
    out1 = bn(x1)
    assert bn.num_batches_tracked.item() == 1, "train-mode forward must advance num_batches_tracked"
    assert not torch.allclose(bn.running_mean, torch.zeros(channels)), "running_mean must move away from its 0 init after one train-mode batch"
    assert not torch.allclose(bn.running_var, torch.ones(channels)), "running_var must move away from its 1 init after one train-mode batch"
    assert out1.mean().abs().item() < 0.5, "train-mode output should be normalized by the BATCH's own mean, near 0, despite x1's mean=3"
    assert abs(out1.std().item() - 1.0) < 0.3, "train-mode output should be normalized by the BATCH's own std, near 1, despite x1's std=5"

    # 3. A second, very differently-scaled batch must ALSO come out normalized to ~unit
    #    variance -- proving train-mode normalization tracks the CURRENT batch, not wherever
    #    the running average has drifted to after step 2.
    x2 = torch.randn(16, channels, 20) * 50.0 - 20.0  # mean ~-20, std ~50 -- a very different scale from x1
    out2 = bn(x2)
    assert bn.num_batches_tracked.item() == 2
    assert out2.mean().abs().item() < 0.5, "train-mode must re-normalize EVERY batch by its own stats, not drift toward the running average"
    assert abs(out2.std().item() - 1.0) < 0.3

    running_mean_after_train = bn.running_mean.clone()
    running_var_after_train = bn.running_var.clone()

    # 4. eval() mode: forward() must become a PURE function of its input -- running stats
    #    frozen, num_batches_tracked frozen, repeated calls bit-identical.
    bn.eval()
    _ = bn(x2)
    assert torch.equal(bn.running_mean, running_mean_after_train), "eval-mode forward must NOT update running_mean"
    assert torch.equal(bn.running_var, running_var_after_train), "eval-mode forward must NOT update running_var"
    assert bn.num_batches_tracked.item() == 2, "eval-mode forward must NOT advance num_batches_tracked"

    out_eval_a = bn(x2)
    out_eval_b = bn(x2)
    assert torch.equal(out_eval_a, out_eval_b), "eval-mode output must be deterministic -- it uses frozen running stats, not this batch's own"

    # 5. The strongest version of the eval check: not just "deterministic", but numerically
    #    equal to the documented frozen-running-stats formula itself --
    #    (x - running_mean) / sqrt(running_var + eps) * gamma + beta.
    rm = running_mean_after_train.view(1, channels, 1)
    rv = running_var_after_train.view(1, channels, 1)
    gamma = bn.weight.view(1, channels, 1)
    beta = bn.bias.view(1, channels, 1)
    expected = (x2 - rm) / torch.sqrt(rv + bn.eps) * gamma + beta
    assert torch.allclose(out_eval_a, expected, atol=1e-5), "eval-mode output must match the documented frozen-running-stats formula exactly"

    # 6. The real-world edge case this architecture must get right before it is trusted in
    #    train.ipynb -- and an assumption worth testing explicitly rather than taking on
    #    faith: for a 3D (N, C, L) input, BatchNorm1d computes each channel's statistics over
    #    N*L elements combined, NOT over N alone. So a train-mode batch of size 1 is actually
    #    FINE here as long as the window length L > 1 (which it always is in this project --
    #    e.g. L=50) -- there is no hidden need for `drop_last=True` on the train loader. The
    #    real failure mode is N*L == 1, i.e. batch_size=1 AND window length L=1 together. This
    #    distinction matters enough to test both sides rather than assume either one.
    bn_fresh = nn.BatchNorm1d(channels)
    bn_fresh.train()
    out_bs1_normal_length = bn_fresh(torch.randn(1, channels, 20))  # batch=1, L=20 -- N*L=20 elements/channel -- must NOT raise
    assert out_bs1_normal_length.shape == (1, channels, 20)

    bn_fresh2 = nn.BatchNorm1d(channels)
    bn_fresh2.train()
    raised = False
    try:
        bn_fresh2(torch.randn(1, channels, 1))  # batch=1, L=1 -- exactly 1 element/channel -- SHOULD raise
    except ValueError:
        raised = True
    assert raised, "BatchNorm1d in train() mode with batch_size=1 AND window length 1 (1 element/channel) should raise ValueError"

    # eval() mode has no such restriction at any size -- it never computes a batch variance at all, only reads the frozen running stats.
    bn_fresh2.eval()
    out_bs1_eval = bn_fresh2(torch.randn(1, channels, 1))  # must not raise, even at the size that raised in train() mode
    assert out_bs1_eval.shape == (1, channels, 1)

    print("BatchNorm1d running statistics: PASSED (train mode normalizes per-batch AND mutates running stats; "
          "eval mode freezes running stats, is deterministic, and matches the documented formula; "
          "batch_size=1 is fine in train() mode when window length > 1 -- N*L elements/channel is what matters, "
          "not N alone -- and only the true 1-element/channel case raises)")


_test_batchnorm_running_statistics_train_vs_eval()



Conv1dSamePadding: PASSED (length preserved for K=3,5,7,9; even K correctly rejected)
Conv1dSamePadding conv_dropout: PASSED (default is a no-op, eval() disables it, train() actually zeroes activations)
Conv1dSamePadding batchnorm wiring: PASSED (default is nn.Identity with zero added state; enabled layer is BatchNorm1d(out_channels))
BatchNorm1d running statistics: PASSED (train mode normalizes per-batch AND mutates running stats; eval mode freezes running stats, is deterministic, and matches the documented formula; batch_size=1 is fine in train() mode when window length > 1 -- N*L elements/channel is what matters, not N alone -- and only the true 1-element/channel case raises)


## 3. `ChannelAttentionModule` (CAM) — learn which feature *channels* matter

Per Section 2 of `ma1dcnn_architecture_plan.docx`: squeeze each
channel's whole timeseries down to one number (global average pooling),
pass all channels through a small bottleneck MLP, then sigmoid to get
one weight per channel in `(0, 1)` — and reweight the original feature
map by that per-channel weight. This is the standard squeeze-and-excite
pattern, applied to N-CMAPSS's sensor channels instead of image color
channels.

In [5]:
class ChannelAttentionModule(nn.Module):
    """CAM: one learned attention weight per CHANNEL, shared across all timesteps."""

    def __init__(self, n_channels, reduction_ratio=CAM_REDUCTION_RATIO):
        super().__init__()

        hidden = max(1, n_channels // reduction_ratio)  # bottleneck width; max(1, ...) guards tiny n_channels from hitting 0

        self.squeeze = nn.AdaptiveAvgPool1d(1)  # collapses the L dimension to 1: (batch, C, L) -> (batch, C, 1)
        self.excite = nn.Sequential(
            nn.Linear(n_channels, hidden),  # C -> C/r bottleneck: forces the network to compress before scoring channels
            nn.ReLU(),                      # nonlinearity inside the bottleneck
            nn.Linear(hidden, n_channels),  # C/r -> C: back to one raw score per channel
            nn.Sigmoid(),                   # squashes every channel's score into (0, 1) -- a per-channel "how much to keep" gate
        )

    def forward(self, x):
        # x: (batch, C, L) -- the feature map coming out of the conv stack.
        batch, channels, length = x.shape  # unpack shape once, reused below

        squeezed = self.squeeze(x)                  # (batch, C, 1) -- one number per channel, averaged over all L timesteps
        squeezed = squeezed.view(batch, channels)    # (batch, C) -- nn.Linear needs a flat feature dim, not a trailing size-1 axis

        channel_weights = self.excite(squeezed)                      # (batch, C) -- one weight in (0, 1) per channel
        channel_weights = channel_weights.view(batch, channels, 1)   # (batch, C, 1) -- reshaped so it broadcasts over L below

        return x * channel_weights  # (batch, C, L) * (batch, C, 1) broadcasts across every timestep: same weight for a channel at every L

In [6]:
def _test_channel_attention_shape_and_gating():
    batch, channels, length = 4, 128, 20
    x = torch.randn(batch, channels, length)

    cam = ChannelAttentionModule(n_channels=channels, reduction_ratio=4)
    out = cam(x)

    assert out.shape == x.shape, out.shape  # CAM reweights, it never resizes

    # Recompute the channel weights the same way forward() does, so this
    # test checks the actual RESULT ("channel c scaled by one number for
    # every timestep"), not just that the shapes happen to line up.
    with torch.no_grad():
        squeezed = cam.squeeze(x).view(batch, channels)
        weights = cam.excite(squeezed)  # (batch, C), each entry expected in (0, 1)

        assert torch.all(weights >= 0) and torch.all(weights <= 1), (weights.min().item(), weights.max().item())

        # Pick one (batch, channel) pair and confirm EVERY timestep in
        # that channel was scaled by the exact same weight.
        b, c = 0, 5
        expected = x[b, c, :] * weights[b, c]
        assert torch.allclose(out[b, c, :], expected, atol=1e-5)

    print("ChannelAttentionModule: PASSED (shape preserved, weights in (0,1), per-channel scaling confirmed)")


_test_channel_attention_shape_and_gating()

ChannelAttentionModule: PASSED (shape preserved, weights in (0,1), per-channel scaling confirmed)


## 4. `ElementwiseAttentionModule` (EAM) — learn which *timesteps* matter

Per Section 2 of `ma1dcnn_architecture_plan.docx`: the complementary
view to CAM. Instead of one weight per channel shared across time, EAM
computes one attention weight **per timestep**, shared across all
channels. A fault signature is often only visible in a few cycles near
failure; EAM lets the network focus there instead of averaging every
timestep equally.

In [7]:
class ElementwiseAttentionModule(nn.Module):
    """EAM: one learned attention weight per TIMESTEP, shared across all channels."""

    def __init__(self, n_channels):
        super().__init__()

        # A 1x1 conv collapses all C channels down to a single score per
        # timestep -- kernel_size=1 means it looks at one timestep at a
        # time, combining information ACROSS channels only; mixing
        # across TIME already happened in the conv blocks before this.
        self.score = nn.Conv1d(in_channels=n_channels, out_channels=1, kernel_size=1)
        self.activation = nn.Sigmoid()  # squashes each timestep's score into (0, 1) -- a per-timestep "how much to keep" gate

    def forward(self, x):
        # x: (batch, C, L) -- the same feature-map shape CAM operates on.
        attention = self.score(x)               # (batch, 1, L) -- one raw score per timestep, all channels combined
        attention = self.activation(attention)  # (batch, 1, L) -- each value now in (0, 1)

        return x * attention  # (batch, C, L) * (batch, 1, L) broadcasts across every channel: same weight for all C at a given timestep

In [8]:
def _test_elementwise_attention_shape_and_sharing():
    batch, channels, length = 4, 128, 20
    x = torch.randn(batch, channels, length)

    eam = ElementwiseAttentionModule(n_channels=channels)
    out = eam(x)

    assert out.shape == x.shape, out.shape  # EAM reweights, it never resizes

    with torch.no_grad():
        attention = eam.activation(eam.score(x))  # (batch, 1, L), recomputed independently of forward()
        assert torch.all(attention >= 0) and torch.all(attention <= 1)

        # The defining property of EAM (vs CAM): at a FIXED timestep, the
        # weight must be identical across every channel. Multiply (not
        # divide, to avoid instability near x==0) and compare directly.
        b, t = 0, 7
        expected = x[b, :, t] * attention[b, 0, t]  # same scalar weight applied to every one of the 128 channels
        assert torch.allclose(out[b, :, t], expected, atol=1e-5)

    print("ElementwiseAttentionModule: PASSED (shape preserved, weights in (0,1), per-timestep sharing across channels confirmed)")


_test_elementwise_attention_shape_and_sharing()

ElementwiseAttentionModule: PASSED (shape preserved, weights in (0,1), per-timestep sharing across channels confirmed)


## 5. `MA1DCNN` — the full architecture

3 same-padded conv blocks → CAM → EAM → global average pooling →
dropout → fully-connected layer → 5-unit output layer. Per Akindoju
(2025), Section 3.4.1 and Table 4.4, and diagrammed in Section 2 of
`ma1dcnn_architecture_plan.docx`.

**On the output layer:** `forward()` returns **raw logits**, not
sigmoid probabilities. `ma1dcnn_architecture_plan.docx` Section 5 found
that Table 4.4's `Use Focal: FALSE` means training should use plain
binary cross-entropy — and PyTorch's numerically-stable implementation
of that, `nn.BCEWithLogitsLoss`, expects raw logits, not
already-sigmoided probabilities, as its input. Keeping `forward()`
logit-valued means this class plugs directly into that loss function
once `training/train.py` is built. `predict_proba()` below applies
sigmoid separately, for whenever an actual probability is wanted (e.g.
evaluation-time thresholding).

In [9]:
class MA1DCNN(nn.Module):
    """3 conv blocks -> CAM -> EAM -> global average pooling -> FC head -> 5 logits."""

    def __init__(self, n_input_channels, channel_1, channel_2, channel_3,
                 kernel_size, fc_units, dropout, n_outputs,
                 cam_reduction_ratio=CAM_REDUCTION_RATIO, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()

        # Three same-padded conv blocks, each widening the channel
        # dimension while leaving the sequence length L untouched (see
        # Conv1dSamePadding above and Section 3.1's proof).
        # conv_dropout and use_batchnorm both default to off -- a no-op -- so every existing
        # caller (BEFORE_TUNING_CONFIG, AFTER_TUNING_CONFIG, BEFORE_TUNING_CONFIG_CONV_DROPOUT)
        # builds a byte-identical model to before either parameter existed. They are two
        # independent, composable levers -- use_batchnorm=True is applied LAYERED ON TOP of
        # whatever conv_dropout rate is also set, never one replacing the other.
        self.conv1 = Conv1dSamePadding(n_input_channels, channel_1, kernel_size, conv_dropout, use_batchnorm)  # C_in -> channel_1 (e.g. 28 -> 32)
        self.conv2 = Conv1dSamePadding(channel_1, channel_2, kernel_size, conv_dropout, use_batchnorm)         # channel_1 -> channel_2 (e.g. 32 -> 64/128)
        self.conv3 = Conv1dSamePadding(channel_2, channel_3, kernel_size, conv_dropout, use_batchnorm)         # channel_2 -> channel_3 (e.g. -> 128)

        # Attention modules operate on the FINAL conv block's output only
        # (channel_3 channels) -- matching the architecture-plan diagram,
        # which places CAM and EAM after all three conv blocks, not in
        # between them.
        self.cam = ChannelAttentionModule(channel_3, reduction_ratio=cam_reduction_ratio)  # learns which channels matter
        self.eam = ElementwiseAttentionModule(channel_3)                                   # learns which timesteps matter

        # Global Average Pooling collapses the time dimension L to a
        # single value per channel -- the ONLY place in this architecture
        # where L is reduced, so the FC layer below never depends on how
        # long the input window was.
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        self.dropout = nn.Dropout(p=dropout)                # zeroes activations at rate p, TRAINING ONLY (Section 3.4)
        self.fc = nn.Linear(channel_3, fc_units)             # channel_3 -> fc_units (e.g. 128 -> 64 or 128 -> 16)
        self.fc_activation = nn.ReLU()                       # nonlinearity between the FC layer and the output layer
        self.output_layer = nn.Linear(fc_units, n_outputs)   # fc_units -> 5: one raw logit per fault component

    def forward(self, x):
        # x: (batch, n_input_channels, L) -- a batch of windowed,
        # already-normalized sensor sequences (once the windowing
        # notebook exists to produce them from preprocessing.py's output).
        x = self.conv1(x)  # (batch, channel_1, L)
        x = self.conv2(x)  # (batch, channel_2, L)
        x = self.conv3(x)  # (batch, channel_3, L) -- L still unchanged, by construction

        x = self.cam(x)  # (batch, channel_3, L) -- channels reweighted by learned importance
        x = self.eam(x)  # (batch, channel_3, L) -- timesteps reweighted by learned importance

        x = self.global_avg_pool(x)   # (batch, channel_3, 1) -- the only step that collapses L
        x = x.view(x.shape[0], -1)    # (batch, channel_3) -- flatten the trailing size-1 axis for nn.Linear

        x = self.dropout(x)          # (batch, channel_3) -- randomly zeroes entries during training only; identity at eval time
        x = self.fc(x)               # (batch, fc_units)
        x = self.fc_activation(x)    # (batch, fc_units)

        logits = self.output_layer(x)  # (batch, n_outputs) -- RAW logits, no sigmoid here (see class docstring above)
        return logits

    def predict_proba(self, x):
        """Return per-fault-component probabilities in [0, 1], not raw logits.

        Applies sigmoid OUTSIDE forward() -- BCEWithLogitsLoss expects
        raw logits for numerical stability, so forward() must stay
        logit-valued during training. This is what evaluation code
        (Section 4.4.3-style thresholding) should call instead of
        forward() directly.
        """
        with torch.no_grad():             # inference-only convenience -- never used inside a training loop's backward pass
            logits = self.forward(x)
            return torch.sigmoid(logits)  # squashes each of the 5 outputs INDEPENDENTLY into (0,1) -- multi-label, not softmax


def build_ma1dcnn(config):
    """Construct an MA1DCNN from a plain hyperparameter dict (e.g. BEFORE_TUNING_CONFIG)."""
    return MA1DCNN(
        n_input_channels=config["n_input_channels"],  # how many sensor channels the windowed input has
        channel_1=config["channel_1"],                # conv block 1's output width
        channel_2=config["channel_2"],                # conv block 2's output width
        channel_3=config["channel_3"],                # conv block 3's output width -- also CAM/EAM's channel count
        kernel_size=config["kernel_size"],             # shared kernel size for all 3 conv blocks
        fc_units=config["fc_units"],                   # width of the hidden fully-connected layer
        dropout=config["dropout"],                     # dropout probability applied before the FC layer
        n_outputs=config["n_outputs"],                 # output width -- fixed at 5 fault components
        # .get() with an off default -- NOT config["conv_dropout"] / config["use_batchnorm"] -- so
        # every config without these keys (BEFORE_TUNING_CONFIG, AFTER_TUNING_CONFIG) keeps building
        # exactly the model it always did.
        conv_dropout=config.get("conv_dropout", 0.0),
        use_batchnorm=config.get("use_batchnorm", False),
    )


## 6. Forward-pass tests — output shape, probability range, and any window length

Two tests: one confirming a single realistic forward pass produces the
right shapes and valid probabilities, and one confirming the model
tolerates *any* input window length without changing its output shape
— the direct, practical consequence of Global Average Pooling being the
only place `L` is ever reduced (Section 3.1 / Section 2 of the
architecture plan).

In [10]:
def _test_ma1dcnn_forward_shape_and_output_ranges():
    torch.manual_seed(0)  # reproducible random weights/inputs -- this test only, not a training run
    config = BEFORE_TUNING_CONFIG
    model = build_ma1dcnn(config)
    model.eval()  # turns dropout OFF so this test is deterministic, not a check of training-mode randomness

    batch, length = 8, 50  # window length is arbitrary here -- Section 3.1 guarantees the conv stack works at any L
    x = torch.randn(batch, config["n_input_channels"], length)

    logits = model(x)
    assert logits.shape == (batch, config["n_outputs"]), logits.shape  # expect (8, 5)

    probs = model.predict_proba(x)
    assert probs.shape == logits.shape
    assert torch.all(probs >= 0) and torch.all(probs <= 1), (probs.min().item(), probs.max().item())

    # Confirm predict_proba is really sigmoid(forward()'s own logits),
    # not silently recomputing something else.
    assert torch.allclose(probs, torch.sigmoid(logits), atol=1e-6)

    print("MA1DCNN forward: PASSED (output shape (batch, 5), probabilities in [0,1], predict_proba matches sigmoid(logits))")


_test_ma1dcnn_forward_shape_and_output_ranges()

MA1DCNN forward: PASSED (output shape (batch, 5), probabilities in [0,1], predict_proba matches sigmoid(logits))


In [11]:
def _test_ma1dcnn_handles_any_window_length():
    torch.manual_seed(1)
    model = build_ma1dcnn(AFTER_TUNING_CONFIG)
    model.eval()

    for length in [10, 50, 200]:  # short, medium, long windows -- none of these should change the output SHAPE
        x = torch.randn(2, AFTER_TUNING_CONFIG["n_input_channels"], length)
        logits = model(x)
        assert logits.shape == (2, AFTER_TUNING_CONFIG["n_outputs"]), (length, logits.shape)

    print("MA1DCNN: PASSED (output shape independent of input window length -- Global Average Pooling working as intended)")


_test_ma1dcnn_handles_any_window_length()

MA1DCNN: PASSED (output shape independent of input window length -- Global Average Pooling working as intended)


### BatchNorm1d integration -- the full `MA1DCNN` model, not just the isolated layer

The test above (Section 2) proves `BatchNorm1d`'s train/eval contract in isolation. This
one proves the WIRING holds up through the rest of the real model -- CAM, EAM, global
average pooling, and the FC head all sit downstream of the 3 conv blocks that now
optionally contain a `BatchNorm1d` each -- and that `BEFORE_TUNING_CONFIG` /
`AFTER_TUNING_CONFIG` (neither of which has a `use_batchnorm` key) are completely
unaffected by this change.

In [12]:
def _test_ma1dcnn_batchnorm_integration_and_default_config_unaffected():
    torch.manual_seed(2)

    # 1. Regression: configs with no "use_batchnorm" key must build a model whose conv
    #    blocks' .bn submodules are nn.Identity -- zero new parameters, zero new buffers,
    #    identical to the model this notebook built before this change.
    default_model = build_ma1dcnn(BEFORE_TUNING_CONFIG)
    for conv in (default_model.conv1, default_model.conv2, default_model.conv3):
        assert isinstance(conv.bn, nn.Identity), type(conv.bn)

    # 2. The new config: conv_dropout AND use_batchnorm are BOTH active at once (layered,
    #    not one replacing the other) -- confirm every conv block actually got a real
    #    BatchNorm1d sized to its own OUT_channels (32, then channel_2, then channel_3 --
    #    NOT all the same number, which would silently indicate they share one instance).
    bn_model = build_ma1dcnn(BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN)
    assert isinstance(bn_model.conv1.bn, nn.BatchNorm1d) and bn_model.conv1.bn.num_features == BEFORE_TUNING_CONFIG["channel_1"]
    assert isinstance(bn_model.conv2.bn, nn.BatchNorm1d) and bn_model.conv2.bn.num_features == BEFORE_TUNING_CONFIG["channel_2"]
    assert isinstance(bn_model.conv3.bn, nn.BatchNorm1d) and bn_model.conv3.bn.num_features == BEFORE_TUNING_CONFIG["channel_3"]
    assert bn_model.conv1.bn is not bn_model.conv2.bn is not bn_model.conv3.bn, "each conv block must own its own BatchNorm1d, not share one"
    assert bn_model.conv1.conv_dropout.p == 0.2, "use_batchnorm=True must not have clobbered conv_dropout -- the two are independent levers"

    # 3. End-to-end shape + probability-range check, same style as Section 6's tests above,
    #    run through the FULL model (conv stack -> CAM -> EAM -> GAP -> FC head) with BN
    #    enabled -- not just the isolated Conv1dSamePadding layer.
    config = BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN
    batch, length = 8, 50  # an ordinary batch -- batch_size=1 is separately covered above and is fine here since length (L) > 1
    x = torch.randn(batch, config["n_input_channels"], length)

    bn_model.train()  # train() mode: BatchNorm1d computes batch stats AND mutates running stats as a side effect
    logits_train = bn_model(x)
    assert logits_train.shape == (batch, config["n_outputs"]), logits_train.shape
    running_mean_snapshot = bn_model.conv1.bn.running_mean.clone()
    assert not torch.allclose(running_mean_snapshot, torch.zeros_like(running_mean_snapshot)), (
        "a train-mode forward pass through the full model must have moved conv1's running_mean away from its 0 init"
    )

    bn_model.eval()  # eval() mode: conv_dropout AND BatchNorm1d's batch-stat path both turn off; predict_proba must be deterministic
    probs_a = bn_model.predict_proba(x)
    probs_b = bn_model.predict_proba(x)
    assert torch.equal(probs_a, probs_b), "eval() mode must be fully deterministic end-to-end, same contract as every other MA1DCNN config"
    assert torch.all(probs_a >= 0) and torch.all(probs_a <= 1), (probs_a.min().item(), probs_a.max().item())
    assert torch.equal(bn_model.conv1.bn.running_mean, running_mean_snapshot), "eval-mode forward must NOT further update conv1's running_mean"

    print("MA1DCNN batchnorm integration: PASSED (default config untouched; BN-enabled config wires a distinct "
          "BatchNorm1d per conv block, composes correctly with conv_dropout, and the full model's train/eval "
          "contract -- mutate-then-freeze running stats, deterministic eval -- holds end to end)")


_test_ma1dcnn_batchnorm_integration_and_default_config_unaffected()


MA1DCNN batchnorm integration: PASSED (default config untouched; BN-enabled config wires a distinct BatchNorm1d per conv block, composes correctly with conv_dropout, and the full model's train/eval contract -- mutate-then-freeze running stats, deterministic eval -- holds end to end)


## 7. Parameter-count check — code must match the hand-derived math

`ma1dcnn_architecture_plan.docx` Section 3.2 derived the conv and
fully-connected parameter counts by hand, using
`params = (K x C_in x C_out) + C_out` for a conv layer and
`params = inputs x outputs + outputs` for a linear layer. This test
recomputes those same formulas and checks PyTorch's own
`.numel()`-counted parameters match **exactly**, for both the
before-tuning and after-tuning configs.

In [13]:
def _conv_param_count(kernel_size, c_in, c_out):
    """params = (K x C_in x C_out) + C_out -- Section 3.2's formula, reused here (not retyped) so the docx and this test can never silently drift apart."""
    return (kernel_size * c_in * c_out) + c_out


def _linear_param_count(n_in, n_out):
    """params = inputs x outputs + outputs -- the same formula, specialized to a fully-connected layer."""
    return (n_in * n_out) + n_out


def _test_param_count_matches_architecture_plan_formula():
    for config, label in [(BEFORE_TUNING_CONFIG, "before tuning"), (AFTER_TUNING_CONFIG, "after tuning")]:
        model = build_ma1dcnn(config)

        # --- Conv stack: sum the formula's prediction for all 3 blocks ---
        expected_conv_total = (
            _conv_param_count(config["kernel_size"], config["n_input_channels"], config["channel_1"])
            + _conv_param_count(config["kernel_size"], config["channel_1"], config["channel_2"])
            + _conv_param_count(config["kernel_size"], config["channel_2"], config["channel_3"])
        )
        # Sum PyTorch's ACTUAL parameter counts for the same 3 conv layers.
        actual_conv_total = (
            sum(p.numel() for p in model.conv1.conv.parameters())
            + sum(p.numel() for p in model.conv2.conv.parameters())
            + sum(p.numel() for p in model.conv3.conv.parameters())
        )
        assert actual_conv_total == expected_conv_total, (label, actual_conv_total, expected_conv_total)

        # --- FC + output layer: same formula, specialized to linear layers ---
        expected_fc_total = (
            _linear_param_count(config["channel_3"], config["fc_units"])
            + _linear_param_count(config["fc_units"], config["n_outputs"])
        )
        actual_fc_total = (
            sum(p.numel() for p in model.fc.parameters())
            + sum(p.numel() for p in model.output_layer.parameters())
        )
        assert actual_fc_total == expected_fc_total, (label, actual_fc_total, expected_fc_total)

        print(f"{label}: conv params {actual_conv_total:,} (== formula {expected_conv_total:,}), "
              f"fc+output params {actual_fc_total:,} (== formula {expected_fc_total:,})")

    print("param count formula: PASSED (PyTorch's actual parameter counts match Section 3.2's hand-derived formula exactly)")


_test_param_count_matches_architecture_plan_formula()

before tuning: conv params 78,176 (== formula 78,176), fc+output params 8,581 (== formula 8,581)
after tuning: conv params 107,168 (== formula 107,168), fc+output params 2,149 (== formula 2,149)
param count formula: PASSED (PyTorch's actual parameter counts match Section 3.2's hand-derived formula exactly)


## 8. Receptive-field check — confirm `RF = 1 + 3 x (K - 1)` empirically

Section 3.3 of the architecture plan derives the receptive field of 3
stacked stride-1 conv layers by formula: 19 timesteps for `K=7`
(before tuning), 13 timesteps for `K=5` (after tuning). This test
proves that formula empirically: perturb exactly ONE input timestep by
a huge amount and check exactly which output timestep positions of the
3rd conv layer actually change.

**Why ReLU is left out for this one test only:** a fully linear
3-layer conv stack's receptive field is exactly `1 + 3(K-1)`, but ReLU
can zero out an activation that a perturbation reached, making that
position look "unaffected" even though the underlying linear
computation did depend on it. Using un-activated conv layers here
isolates the pure structural receptive field Section 3.3 is about, kept
separate from `MA1DCNN`'s own (ReLU-activated) `Conv1dSamePadding`
blocks used everywhere else in this notebook.

In [14]:
def _test_receptive_field_matches_formula():
    for kernel_size, expected_rf in [(7, 19), (5, 13)]:
        length = 41    # long enough that the receptive field doesn't reach either edge from the center position
        center = length // 2
        padding = (kernel_size - 1) // 2  # same-padding formula from Section 3.1, reused here for the raw nn.Conv1d layers below

        # Three stacked, UNACTIVATED conv layers (no ReLU) -- see the
        # markdown above for why activation is deliberately left out
        # for this one test.
        conv1 = nn.Conv1d(1, 4, kernel_size, padding=padding)
        conv2 = nn.Conv1d(4, 4, kernel_size, padding=padding)
        conv3 = nn.Conv1d(4, 1, kernel_size, padding=padding)

        x_base = torch.randn(1, 1, length)
        x_perturbed = x_base.clone()
        x_perturbed[0, 0, center] += 100.0  # a huge, unmistakable change at exactly one input timestep

        with torch.no_grad():
            out_base = conv3(conv2(conv1(x_base)))
            out_perturbed = conv3(conv2(conv1(x_perturbed)))

        diff = (out_base - out_perturbed).abs().squeeze()      # (length,) -- how much each output position changed
        changed_positions = (diff > 1e-4).nonzero().flatten()  # indices of positions that changed at all

        actual_rf = int(changed_positions.max() - changed_positions.min()) + 1  # span of affected positions, inclusive
        assert actual_rf == expected_rf, (kernel_size, actual_rf, expected_rf)

        print(f"K={kernel_size}: empirical receptive field = {actual_rf} timesteps (formula predicts {expected_rf}) -- PASSED")

    print("receptive field formula: PASSED (RF = 1 + 3(K-1) confirmed empirically, not just derived on paper)")


_test_receptive_field_matches_formula()

K=7: empirical receptive field = 19 timesteps (formula predicts 19) -- PASSED
K=5: empirical receptive field = 13 timesteps (formula predicts 13) -- PASSED
receptive field formula: PASSED (RF = 1 + 3(K-1) confirmed empirically, not just derived on paper)


## 9. Model summary for both configs (for reading, not a test)

A plain printout — not a test — showing the total trainable parameter
count for the full `MA1DCNN` (including CAM/EAM, which the formula
check above deliberately does not cover, since CAM/EAM's own parameter
counts were not part of Section 3.2's worked example). Useful for
eyeballing how much the whole model's size differs between the
before-tuning and after-tuning hyperparameters.

In [15]:
def _print_model_summary(config, label):
    model = build_ma1dcnn(config)
    total_params = sum(p.numel() for p in model.parameters())          # every learnable number in the whole model
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # same here, since nothing is frozen

    print(f"{label}:")
    print(f"    kernel_size={config['kernel_size']}, channels={config['channel_1']}/{config['channel_2']}/{config['channel_3']}, "
          f"fc_units={config['fc_units']}, dropout={config['dropout']}")
    print(f"    total parameters:     {total_params:,}")
    print(f"    trainable parameters: {trainable_params:,}")


_print_model_summary(BEFORE_TUNING_CONFIG, "Before tuning (Table 4.4)")
print()
_print_model_summary(AFTER_TUNING_CONFIG, "After tuning (Table 4.4)")

Before tuning (Table 4.4):
    kernel_size=7, channels=32/64/128, fc_units=64, dropout=0.5
    total parameters:     95,238
    trainable parameters: 95,238

After tuning (Table 4.4):
    kernel_size=5, channels=32/128/128, fc_units=16, dropout=0.39919
    total parameters:     117,798
    trainable parameters: 117,798


## What's next

`MA1DCNN` is now built and verified against synthetic data -- correct output shape,
correct probability range, correct parameter counts, a confirmed receptive field, a
confirmed no-op-by-default conv-stack dropout path, and (new) a confirmed, no-op-by-
default `BatchNorm1d` path with its own dedicated train/eval running-statistics test --
and has been run for real against the actual windowed N-CMAPSS data in `train.ipynb`
(Section 10), across four configs so far: plain `BEFORE_TUNING_CONFIG` (immediate
overfitting, best `val_f1_macro=0.7995` across two runs), `weight_decay` raised 10x
(best `val_f1_macro=0.8212` across two runs, gap cut substantially but not eliminated),
AdamW's decoupled weight_decay (worse than both, on two independent runs), and
`BEFORE_TUNING_CONFIG_CONV_DROPOUT` + `TRAIN_REGULARIZED_CONFIG` (best
`val_f1_macro=0.8177`, generalization gap cut to 0.0288 -- the tightest of any run so
far; see `overfitting_adamw_comparison.xlsx`).

This notebook now also adds **BatchNorm1d after each conv block**
(`BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN`, Section 1) -- `self.bn`, wired BEFORE the
activation inside `Conv1dSamePadding`, LAYERED ON TOP of conv-stack dropout rather than
replacing it. `use_batchnorm` defaults to `False` everywhere else, so
`BEFORE_TUNING_CONFIG`, `AFTER_TUNING_CONFIG`, and `BEFORE_TUNING_CONFIG_CONV_DROPOUT`
are all unaffected -- their `conv1.bn`/`conv2.bn`/`conv3.bn` submodules are
`nn.Identity()`, with zero added parameters or buffers. Section 2's new test proves the
train/eval running-statistics contract in isolation (train mode normalizes by, and
mutates toward, the CURRENT batch's own stats; eval mode freezes and reuses the running
stats, deterministically, matching the documented formula exactly); the new integration
test after Section 6 proves the same contract holds through the full model -- CAM, EAM,
global average pooling, and the FC head all downstream of it -- and that it composes
correctly with conv_dropout rather than clobbering it.

Remaining work, in order:

1. **Sync this change into `train.ipynb`** -- add a new optional Section 10 cell using
   `BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN` with `TRAIN_REGULARIZED_CONFIG`, run it for
   real (at least twice, given this project's documented CPU run-to-run variance), and
   compare its generalization gap against every run above. The batch_size=1 question this
   raised while testing was checked directly, not assumed: BatchNorm1d normalizes a 3D
   `(N, C, L)` input over `N*L` elements per channel, not `N` alone, so a final
   training-mode batch of size 1 is fine here as long as the window length `L > 1` (which
   it always is) -- no `drop_last=True` needed on the train loader.
2. **Run Section 10 again with `AFTER_TUNING_CONFIG` / `TRAIN_AFTER_TUNING_CONFIG`** and
   compare against everything above -- a ceiling check (it changes the architecture AND
   every training hyperparameter at once), not a diagnostic.
3. **`training/optuna_search.py`** -- the hyperparameter search that reuses
   `run_training`'s own `model_config`/`train_config` arguments to reproduce, or improve
   on, Table 4.4's tuning result systematically rather than by hand.

Only once a tuned, non-overfitting baseline is trained here should any of this be ported
into `src/edge_fault_dx/models/ma1dcnn.py`, following the same "prove it standalone
first" discipline every notebook in this project has been built under.
